# NiOA-DRNN — Multi-Horizon Auto-Loop Training

**Author** : Anwesha Singh  
**Dept.**   : Computer Science Engineering, Manipal University Jaipur

---

## How to use

| Cell | What to do |
|---|---|
| **Cell 1** | Run once per kernel session |
| **Cell 2** | Set `HORIZONS_TO_RUN` — then never touch again |
| **Cell 3** | Run and walk away — all horizons train automatically |
| **Cell 4** | Run after Cell 3 to print the cross-horizon table |

## Why GPU works now (important for your report)

Your training set is **(1,874,617 × 120 × 17) float32 ≈ 15 GB**.  
Two standard approaches fail at this scale:

- `model.fit(X_train, y_train)` — Keras converts the entire 15 GB numpy array  
  into a GPU-resident EagerTensor before the first batch runs → OOM crash.
- `tf.data.Dataset.from_tensor_slices((X, y))` — same problem; the full  
  array becomes a graph constant in GPU memory → crash.

The correct solution is **`tf.keras.utils.Sequence`** (`SequenceDataGenerator`  
in `core/utils.py`).  Keras calls `__getitem__(i)` to fetch one numpy batch  
(32 × 120 × 17 ≈ 1.3 MB) and converts only **that batch** to a GPU tensor.  
The 15 GB array never leaves CPU RAM as a whole.  Peak GPU usage = one batch.


## Cell 1 — Imports and Environment

In [1]:
import os, sys, json, gc, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')    # non-interactive backend — required for unattended loops
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import tensorflow as tf
import tensorflow.keras.backend as K
from datetime import datetime
from tensorflow.keras.callbacks import EarlyStopping

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
for _p in [PROJECT_ROOT, os.path.join(PROJECT_ROOT, 'core')]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

from core.config        import (
    DATA_PATH, SPLITS_DIR, NIOA_RESULTS_DIR, RESULTS_DIR,
    FORECAST_HORIZONS, SEQUENCE_LENGTH, TRAIN_RATIO, VAL_RATIO,
    RANDOM_SEED, N_AGENTS, MAX_ITERATIONS, EXPLORATION_FACTOR,
    EXPLOITATION_FACTOR, OPT_SUBSET_RATIO, OPT_EPOCHS, OPT_PATIENCE,
    OPT_TIME_LIMIT, FINAL_EPOCHS, FINAL_PATIENCE, HYPERPARAMETER_BOUNDS,
    GAP_TOLERANCE_FACTOR,
)
from core.utils         import (
    set_seeds, configure_gpu, scale_numeric_features,
    create_sequences, to_python_types, SequenceDataGenerator,
)
from core.preprocessing import load_and_prepare_data, split_by_timestamp
from core.models        import NinjaOptimizationAlgorithm, create_lstm_model
from core.train         import objective_function_lstm
from core.evaluate      import evaluate_keras_model

# configure_gpu() MUST be called before any TF computation or model creation
configure_gpu()
set_seeds(RANDOM_SEED)
sns.set(style='whitegrid')

ANALYSIS_DIR = os.path.join(RESULTS_DIR, 'analysis')
os.makedirs(ANALYSIS_DIR, exist_ok=True)

print(f'TensorFlow   : {tf.__version__}')
print(f'Project root : {PROJECT_ROOT}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPU(s) found : {len(gpus)}  →  {[g.name for g in gpus]}')

GPU ready — 1 device(s), memory growth enabled.
TensorFlow   : 2.10.1
Project root : C:\Users\AnweshaSingh\anaconda_projects\pipelinev5
GPU(s) found : 1  →  ['/physical_device:GPU:0']


## Cell 2 — Select Horizons
Edit `HORIZONS_TO_RUN` once.  Then run Cell 3 and do not touch this notebook again.

In [2]:
# ── Only edit these two lines ────────────────────────────────────────────────
HORIZONS_TO_RUN = [60, 300, 900, 1800]
SKIP_COMPLETED  = False   # True = skip horizons that already have a saved .h5
# ────────────────────────────────────────────────────────────────────────────

# Batch size for final training.  32 sends one 32×120×17 = 1.3 MB batch to GPU.
# Only reduce this if your GPU has less than ~4 GB VRAM.
FINAL_BATCH_SIZE = 16

for _h in HORIZONS_TO_RUN:
    assert _h in FORECAST_HORIZONS, f'{_h} not in {FORECAST_HORIZONS}'

print(f'Horizons to train : {HORIZONS_TO_RUN}')
print(f'Batch size        : {FINAL_BATCH_SIZE}')
print(f'Skip completed    : {SKIP_COMPLETED}')

Horizons to train : [60, 300, 900, 1800]
Batch size        : 16
Skip completed    : False


## Cell 3 — Full Pipeline Loop

For each horizon this cell runs:

```
 3a  Load + gap-aware preprocess
 3b  Chronological split (train 70% / val 15% / test 15%) + leakage check
 3c  Feature scaling (train-only scaler) + stride-trick sequence generation
 3d  Save canonical splits  →  results/splits/horizon_{k}/
 3e  NiOA hyperparameter search on 30% subset via SequenceDataGenerator
 3f  Final model training on full train split via SequenceDataGenerator
 3g  Save model (.h5), scaler, training_config.json
 3h  Evaluate on held-out test set, save metrics.json + predictions
 3i  Save 5 publication plots  →  results/nioa_drnn/k{k}_*/plots/
     GPU cleared before next horizon
```

In [ ]:
all_results = {}   # filled as each horizon completes
loop_start  = time.time()

for _loop_idx, HORIZON in enumerate(HORIZONS_TO_RUN):

    # ── Banner ──────────────────────────────────────────────────────────
    _elapsed = (time.time() - loop_start) / 3600
    print('\n' + '=' * 65)
    print(f'  Horizon {_loop_idx+1}/{len(HORIZONS_TO_RUN)} : '
          f'k = {HORIZON} s   {datetime.now().strftime("%H:%M:%S")}   '
          f'{_elapsed:.2f} h elapsed')
    print('=' * 65)

    # ── Skip check ──────────────────────────────────────────────────────
    if SKIP_COMPLETED and os.path.isdir(NIOA_RESULTS_DIR):
        _done = [
            d for d in os.listdir(NIOA_RESULTS_DIR)
            if d.startswith(f'k{HORIZON}_') and
            os.path.isfile(os.path.join(
                NIOA_RESULTS_DIR, d, 'model', f'NiOA_DRNN_k{HORIZON}.h5'
            ))
        ]
        if _done:
            print(f'  [SKIP]  model already saved in {_done[-1]}')
            continue

    # ── 3a. Load + preprocess ────────────────────────────────────────────
    print(f'\n[3a] Preprocessing  (k={HORIZON}s, gap_factor={GAP_TOLERANCE_FACTOR}x)')
    df = load_and_prepare_data(
        DATA_PATH, k=HORIZON, gap_factor=GAP_TOLERANCE_FACTOR, verbose=True
    )

    # ── 3b. Split ────────────────────────────────────────────────────────
    print('[3b] Splitting')
    t_tr  = df['server_timestamp'].quantile(TRAIN_RATIO)
    t_va  = df['server_timestamp'].quantile(TRAIN_RATIO + VAL_RATIO)
    train_df, val_df, test_df = split_by_timestamp(df, t_tr, t_va)

    assert train_df['server_timestamp'].max() < val_df['server_timestamp'].min()
    assert val_df['server_timestamp'].max()   < test_df['server_timestamp'].min()

    print(f'  train {len(train_df):>9,}  val {len(val_df):>9,}  '
          f'test {len(test_df):>9,}  — no leakage')

    # ── 3c. Scale + sequences ────────────────────────────────────────────
    print('[3c] Scaling + sequence generation')
    TARGET       = 'energy_delta_k'
    feature_cols = [
        c for c in train_df.columns
        if c not in ['server_timestamp', 'energy', TARGET]
    ]

    X_tr_sc, X_va_sc, X_te_sc, scaler = scale_numeric_features(
        train_df[feature_cols].copy(),
        val_df[feature_cols].copy(),
        test_df[feature_cols].copy(),
        feature_cols,
    )

    # create_sequences uses numpy stride tricks — faster and same peak RAM
    X_train_seq, y_train_seq = create_sequences(
        X_tr_sc.values, train_df[TARGET].values, SEQUENCE_LENGTH)
    X_val_seq,   y_val_seq   = create_sequences(
        X_va_sc.values, val_df[TARGET].values,   SEQUENCE_LENGTH)
    X_test_seq,  y_test_seq  = create_sequences(
        X_te_sc.values, test_df[TARGET].values,  SEQUENCE_LENGTH)

    SEQ_LEN   = X_train_seq.shape[1]
    NUM_FEATS = X_train_seq.shape[2]

    print(f'  features ({len(feature_cols)}): {feature_cols}')
    print(f'  X_train {X_train_seq.shape}  '
          f'X_val {X_val_seq.shape}  '
          f'X_test {X_test_seq.shape}')
    print(f'  RAM (train seq) ≈ '
          f'{X_train_seq.nbytes / 1e9:.2f} GB   '
          f'GPU per batch ≈ '
          f'{FINAL_BATCH_SIZE * SEQ_LEN * NUM_FEATS * 4 / 1e6:.1f} MB')

    # Free the raw dataframes — sequences are all we need from here
    del df, train_df, val_df, test_df, X_tr_sc, X_va_sc, X_te_sc
    gc.collect()

    # ── 3d. Save canonical splits ────────────────────────────────────────
    print('[3d] Saving canonical splits')
    _sdir = os.path.join(SPLITS_DIR, f'horizon_{HORIZON}')
    os.makedirs(_sdir, exist_ok=True)

    np.save(os.path.join(_sdir, 'X_train.npy'), X_train_seq)
    np.save(os.path.join(_sdir, 'y_train.npy'), y_train_seq)
    np.save(os.path.join(_sdir, 'X_val.npy'),   X_val_seq)
    np.save(os.path.join(_sdir, 'y_val.npy'),   y_val_seq)
    np.save(os.path.join(_sdir, 'X_test.npy'),  X_test_seq)
    np.save(os.path.join(_sdir, 'y_test.npy'),  y_test_seq)
    joblib.dump(scaler, os.path.join(_sdir, 'scaler.pkl'))

    with open(os.path.join(_sdir, 'feature_cols.json'), 'w') as _f:
        json.dump(feature_cols, _f, indent=2)
    with open(os.path.join(_sdir, 'split_metadata.json'), 'w') as _f:
        json.dump({
            'horizon_k'         : HORIZON,
            'sequence_length'   : SEQUENCE_LENGTH,
            'gap_tolerance'     : GAP_TOLERANCE_FACTOR,
            'train_ratio'       : TRAIN_RATIO,
            'val_ratio'         : VAL_RATIO,
            'n_train_sequences' : int(len(X_train_seq)),
            'n_val_sequences'   : int(len(X_val_seq)),
            'n_test_sequences'  : int(len(X_test_seq)),
            'n_features'        : int(NUM_FEATS),
            'random_seed'       : RANDOM_SEED,
            'generated_at'      : datetime.now().isoformat(),
        }, _f, indent=2)
    print(f'  splits → {_sdir}')

    # ── 3e. NiOA hyperparameter optimisation ─────────────────────────────
    #
    # Subset size: 30% of training sequences.
    # For k=60 that is ~560K sequences ≈ 4.5 GB — still too large for
    # model.fit(X, y).  objective_function_lstm uses SequenceDataGenerator
    # internally so each NiOA trial also stays GPU-safe.
    # ─────────────────────────────────────────────────────────────────────
    print(f'[3e] NiOA optimisation  '
          f'(agents={N_AGENTS}  iter={MAX_ITERATIONS}  '
          f'subset={int(OPT_SUBSET_RATIO*100)}%  '
          f'limit={OPT_TIME_LIMIT}s/trial)')

    _n_tr = int(len(X_train_seq) * OPT_SUBSET_RATIO)
    _n_va = int(len(X_val_seq)   * OPT_SUBSET_RATIO)

    # Freeze references in default args to avoid late-binding closure bug
    _Xtr = X_train_seq[:_n_tr].copy()
    _ytr = y_train_seq[:_n_tr].copy()
    _Xva = X_val_seq[:_n_va].copy()
    _yva = y_val_seq[:_n_va].copy()

    def _objective(p,
                   Xtr=_Xtr, ytr=_ytr,
                   Xva=_Xva, yva=_yva):
        return objective_function_lstm(
            p, Xtr, ytr, Xva, yva,
            opt_epochs=OPT_EPOCHS, opt_patience=OPT_PATIENCE,
            time_limit=OPT_TIME_LIMIT,
        )

    ninja = NinjaOptimizationAlgorithm(
        objective_function  = _objective,
        bounds              = HYPERPARAMETER_BOUNDS,
        n_agents            = N_AGENTS,
        max_iterations      = MAX_ITERATIONS,
        exploration_factor  = EXPLORATION_FACTOR,
        exploitation_factor = EXPLOITATION_FACTOR,
        verbose             = True,
    )
    best_params, best_loss, convergence = ninja.optimize()

    print(f'  Best NiOA val MSE : {best_loss:.8f}')
    print(f'  Best params : {best_params}')

    del ninja, _Xtr, _ytr, _Xva, _yva
    gc.collect()

    # ── 3f. Experiment directory ─────────────────────────────────────────
    _ts     = datetime.now().strftime('%Y%m%d_%H%M%S')
    exp_dir = os.path.join(
        NIOA_RESULTS_DIR, f'k{HORIZON}_seq{SEQUENCE_LENGTH}_{_ts}'
    )
    for _sub in ['model', 'predictions', 'plots']:
        os.makedirs(os.path.join(exp_dir, _sub), exist_ok=True)

    with open(os.path.join(exp_dir, 'best_params.json'), 'w') as _f:
        json.dump(to_python_types(best_params), _f, indent=4)
    np.save(os.path.join(exp_dir, 'convergence.npy'), np.array(convergence))

    # ── 3f (cont). Final model training ──────────────────────────────────
    #
    # SequenceDataGenerator is used here instead of model.fit(X, y).
    # Each __getitem__ call returns one numpy batch → Keras sends it to GPU
    # → forward pass + backward pass → GPU tensor freed.
    # The 15 GB array never leaves CPU RAM as a whole.
    # ─────────────────────────────────────────────────────────────────────
    print(f'[3f] Final training  '
          f'(batch={FINAL_BATCH_SIZE}  epochs={FINAL_EPOCHS}  '
          f'patience={FINAL_PATIENCE})')
    K.clear_session()
    gc.collect()

    model = create_lstm_model(best_params, SEQ_LEN, NUM_FEATS)
    model.summary()

    train_gen = SequenceDataGenerator(
        X_train_seq, y_train_seq,
        batch_size=FINAL_BATCH_SIZE, shuffle=True, seed=RANDOM_SEED
    )
    val_gen = SequenceDataGenerator(
        X_val_seq, y_val_seq,
        batch_size=FINAL_BATCH_SIZE, shuffle=False
    )

    history = model.fit(
        train_gen,
        validation_data = val_gen,
        epochs          = FINAL_EPOCHS,
        callbacks       = [
            EarlyStopping(
                monitor='val_loss', patience=FINAL_PATIENCE,
                restore_best_weights=True, verbose=1,
            )
        ],
        verbose=1,
    )
    del train_gen, val_gen
    gc.collect()

    # ── 3g. Save model + config ──────────────────────────────────────────
    print('[3g] Saving model')
    model.save(os.path.join(exp_dir, 'model', f'NiOA_DRNN_k{HORIZON}.h5'))
    joblib.dump(scaler, os.path.join(exp_dir, 'scaler.pkl'))
    with open(os.path.join(exp_dir, 'training_config.json'), 'w') as _f:
        json.dump({
            'horizon_k'         : HORIZON,
            'sequence_length'   : SEQUENCE_LENGTH,
            'gap_tolerance'     : GAP_TOLERANCE_FACTOR,
            'train_ratio'       : TRAIN_RATIO,
            'val_ratio'         : VAL_RATIO,
            'random_seed'       : RANDOM_SEED,
            'final_batch_size'  : FINAL_BATCH_SIZE,
            'opt_subset_ratio'  : OPT_SUBSET_RATIO,
            'n_agents'          : N_AGENTS,
            'max_iterations'    : MAX_ITERATIONS,
            'final_epochs'      : FINAL_EPOCHS,
            'final_patience'    : FINAL_PATIENCE,
            'best_params'       : to_python_types(best_params),
            'best_nioa_loss'    : float(best_loss),
            'completed_at'      : datetime.now().isoformat(),
        }, _f, indent=4)

    # ── 3h. Evaluation ───────────────────────────────────────────────────
    print('[3h] Evaluating on test set')
    metrics_df, y_true, y_pred, _note = evaluate_keras_model(
        model, X_test_seq, y_test_seq
    )
    metrics_dict = dict(zip(metrics_df['Metric'], metrics_df['Value']))
    print(metrics_df.to_string(index=False))

    np.save(os.path.join(exp_dir, 'predictions', 'y_test_true.npy'), y_true)
    np.save(os.path.join(exp_dir, 'predictions', 'y_test_pred.npy'), y_pred)
    with open(os.path.join(exp_dir, 'metrics.json'), 'w') as _f:
        json.dump(metrics_dict, _f, indent=4)

    all_results[HORIZON] = {
        'metrics'    : metrics_dict,
        'exp_dir'    : exp_dir,
        'best_params': to_python_types(best_params),
        'best_loss'  : float(best_loss),
    }

    # ── 3i. Plots (all saved to disk — Agg backend, no screen needed) ────
    print('[3i] Saving plots')
    _pd = os.path.join(exp_dir, 'plots')

    # 1. Predicted vs actual
    _fig, _ax = plt.subplots(figsize=(6, 6))
    _ax.scatter(y_true, y_pred, alpha=0.4, s=5, color='steelblue')
    _lim = [min(y_true.min(), y_pred.min()),
             max(y_true.max(), y_pred.max())]
    _ax.plot(_lim, _lim, 'r--', lw=1.5, label='Perfect prediction')
    _ax.set_xlabel(f'Actual ΔE  k={HORIZON}s (kWh)')
    _ax.set_ylabel(f'Predicted ΔE  k={HORIZON}s (kWh)')
    _ax.set_title(f'Predicted vs Actual  k={HORIZON}s')
    _ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(_pd, 'pred_vs_actual.png'), dpi=200)
    plt.close('all')

    # 2. Residual distribution
    _fig, _ax = plt.subplots(figsize=(7, 4))
    sns.histplot(y_true - y_pred, bins=60, kde=True, ax=_ax, color='steelblue')
    _ax.axvline(0, color='red', lw=1.2, linestyle='--')
    _ax.set_xlabel('Residual (kWh)')
    _ax.set_title(f'Residual Distribution  k={HORIZON}s')
    plt.tight_layout()
    plt.savefig(os.path.join(_pd, 'residuals.png'), dpi=200)
    plt.close('all')

    # 3. Training / validation loss
    _fig, _ax = plt.subplots(figsize=(8, 4))
    _ax.plot(history.history['loss'],     lw=1.5, label='Train MSE')
    _ax.plot(history.history['val_loss'], lw=1.5, label='Val MSE')
    _ax.set_xlabel('Epoch');  _ax.set_ylabel('MSE')
    _ax.set_title(f'Training / Validation Loss  k={HORIZON}s')
    _ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(_pd, 'training_curve.png'), dpi=200)
    plt.close('all')

    # 4. NiOA convergence
    _fig, _ax = plt.subplots(figsize=(7, 4))
    _ax.plot(convergence, marker='o', lw=2, color='darkorange', ms=6)
    _ax.set_xlabel('Iteration  (0 = initial population)')
    _ax.set_ylabel('Best validation MSE')
    _ax.set_title(f'NiOA Convergence  k={HORIZON}s')
    plt.tight_layout()
    plt.savefig(os.path.join(_pd, 'nioa_convergence.png'), dpi=200)
    plt.close('all')

    # 5. Time-series overlay (first 500 test samples)
    _ns = min(500, len(y_true))
    _fig, _ax = plt.subplots(figsize=(12, 4))
    _ax.plot(y_true[:_ns], lw=1.0, alpha=0.9, label='Actual')
    _ax.plot(y_pred[:_ns], lw=1.0, alpha=0.75,
             linestyle='--', label='Predicted')
    _ax.set_xlabel('Test sample index')
    _ax.set_ylabel(f'ΔE  k={HORIZON}s (kWh)')
    _ax.set_title(f'First {_ns} test samples  k={HORIZON}s')
    _ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(_pd, 'time_series_overlay.png'), dpi=200)
    plt.close('all')

    print(f'  5 plots → {_pd}')

    # ── End-of-horizon cleanup ───────────────────────────────────────────
    _total_h = (time.time() - loop_start) / 3600
    print(f'\n  DONE k={HORIZON}s  '
          f'MAE={metrics_dict["MAE"]:.6f}  '
          f'RMSE={metrics_dict["RMSE"]:.6f}  '
          f'R2={metrics_dict["R2"]:.4f}  '
          f'[{_total_h:.2f} h total]')
    print('=' * 65)

    del (model, history, X_train_seq, y_train_seq,
         X_val_seq, y_val_seq, X_test_seq, y_test_seq,
         y_true, y_pred, scaler, best_params, convergence)
    K.clear_session()
    gc.collect()

# ── Loop complete ────────────────────────────────────────────────────────────
_total_h = (time.time() - loop_start) / 3600
print(f'\nAll horizons complete — {_total_h:.2f} h total')
if all_results:
    print(f'\n{"k (s)":>8} {"MAE":>12} {"RMSE":>12} {"R2":>8} {"sMAPE":>10}')
    print('-' * 56)
    for _hk, _r in sorted(all_results.items()):
        _m = _r['metrics']
        print(f'{_hk:>8} {_m["MAE"]:>12.6f} '
              f'{_m["RMSE"]:>12.6f} '
              f'{_m["R2"]:>8.4f} '
              f'{_m["sMAPE"]:>10.4f}')


  Horizon 1/4 : k = 60 s   16:21:22   0.00 h elapsed

[3a] Preprocessing  (k=60s, gap_factor=2.0x)

  Preprocessing summary  (k = 60 s, tolerance = 2.0x)
  Rows in raw CSV          :  3,178,051
  Gap-corrupted targets    :     10,454  (elapsed > 120 s)
  Counter-reset targets    :    152,859  (delta_E < 0)
  Dropped for NaN          :    163,314
  Removed by Z-score       :    336,462  (features only)
  Final usable rows        :  2,678,196

  Target  energy_delta_k  statistics:
    mean  =     6.639824 kWh
    std   =    52.195938 kWh
    min   =     0.000000 kWh
    max   =   417.777000 kWh
    |max| =   417.777000 kWh

[3b] Splitting
  train 1,874,737  val   401,729  test   401,730  — no leakage
[3c] Scaling + sequence generation
  features (17): ['weekday', 'voltage', 'current', 'power', 'frequency', 'power_factor', 'sensor_temperature', 'cpu_usage_percent', 'WORKSTATION_CPU_POWER', 'WORKSTATION_CPU_TEMP', 'WORKSTATION_GPU', 'WORKSTATION_GPU_POWER', 'WORKSTATION_GPU_TEMP', 'WORKST

## Cell 4 — Cross-Horizon Summary
Run after Cell 3 completes.

In [ ]:
if not all_results:
    print('No results yet — run Cell 3 first.')
else:
    _rows = [
        {
            'Horizon_k_s'   : _hk,
            'MAE'           : _r['metrics']['MAE'],
            'RMSE'          : _r['metrics']['RMSE'],
            'R2'            : _r['metrics']['R2'],
            'sMAPE'         : _r['metrics']['sMAPE'],
            'NiOA_best_MSE' : _r['best_loss'],
            'exp_dir'       : os.path.basename(_r['exp_dir']),
        }
        for _hk, _r in sorted(all_results.items())
    ]
    summary = pd.DataFrame(_rows)

    print('NiOA-DRNN — all trained horizons')
    print('=' * 72)
    print(summary.to_string(index=False, float_format='{:.6f}'.format))
    print('=' * 72)

    _csv = os.path.join(ANALYSIS_DIR, 'nioa_drnn_all_horizons.csv')
    summary.to_csv(_csv, index=False)
    print(f'\nCSV  → {_csv}')

    # MAE bar chart
    _fig, _ax = plt.subplots(figsize=(8, 4))
    _ax.bar(
        [f'{r["Horizon_k_s"]}s' for r in _rows],
        [r['MAE'] for r in _rows],
        color='steelblue', edgecolor='white',
    )
    _ax.set_xlabel('Forecast horizon k')
    _ax.set_ylabel('MAE (kWh)')
    _ax.set_title('NiOA-DRNN — MAE per horizon')
    plt.tight_layout()
    _bar = os.path.join(ANALYSIS_DIR, 'nioa_mae_per_horizon.png')
    plt.savefig(_bar, dpi=200)
    plt.close('all')
    print(f'Plot → {_bar}')